In [2]:
#Module Imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from PIL import Image, ImageDraw
import gif
import json
from mplsoccer import Pitch

In [3]:
match_id = 3749493
file = f"{match_id}.json"
with open(f"/workspaces/football-scout/data/raw/events/{file}", "r") as file:
    match = json.load(file)
df = pd.DataFrame(match)
print(df)

                                        id  index  period     timestamp  \
0     d78142c2-54c8-48fd-8346-67ab75d01a5e      1       1  00:00:00.000   
1     9815d71d-139d-430d-9c7a-5260db57773f      2       1  00:00:00.000   
2     d9b7e5eb-8235-4295-84e9-db9d84b847ee      3       1  00:00:00.000   
3     c1d423be-aee9-4e1b-a6d5-ec99ed7a0551      4       1  00:00:00.000   
4     729957b9-d479-49ab-995f-608a19177be5      5       1  00:00:00.983   
...                                    ...    ...     ...           ...   
3052  09e27392-1a91-4031-afdc-5ffdbc6805fe   3053       2  00:48:52.181   
3053  ebaf64b0-396f-4947-adf5-8960ca01e98c   3054       2  00:48:52.206   
3054  ac28cf9e-3513-4812-86d5-4870f16aedbf   3055       2  00:48:53.488   
3055  0261ba5b-8248-447c-b5c9-a528b6096dd8   3056       2  00:48:58.967   
3056  d85550fc-b0b0-4b2f-a7c9-03bcd82853a7   3057       2  00:48:58.967   

      minute  second                                 type  possession  \
0          0       0    {'

In [10]:
def markerdecision(event_type):
    markerdict = {"Block":"s","Foul Committed":"x","Dribble":"o","Interception":"D","Clearance":"^"}
    return markerdict[event_type]

def arrowdecision(event_type):
    arrowdict = {"Pass":"->","Clearance":"-","Shot":"fancy"}
    return arrowdict[event_type]

def visualise(match_id):
    file = f"{match_id}.json"
    with open(f"/workspaces/football-scout/data/raw/events/{file}", "r") as file:
        match = json.load(file)
    notable_events = ["Pass",
                      "Clearance",
                      "Block",
                      "Dribble",
                      "Foul Committed",
                      "Interception",
                      "Shot"]
    animation_events = [
        event for event in match
        if event["type"]["name"] in notable_events]
    

    pitch = Pitch(pitch_type="statsbomb")
    fig, ax = pitch.draw()
    
    def update(frame):
        ax.clear()
        pitch.draw(ax=ax)
        event=animation_events[frame]
        event_type = event["type"]["name"]

        if event_type not in notable_events:
            return

        counter = 0
        start = event["location"]

        if event_type == "Pass":
            end = event["pass"]["end_location"]

        elif event_type == "Clearance":
            if "end_location" not in event["clearance"]:
                end = start
                counter = 1
            else:
                end = event["clearance"]["end_location"]

        elif event_type == "Shot":
            end = event["shot"]["end_location"]

        elif event_type in ["Block", "Dribble", "Foul Committed", "Interception"]:
            end = start
        
        if event_type in ["Pass", "Clearance", "Shot"] and counter == 0:
            ax.annotate("",
                        xy=(end[0], end[1]),
                        xytext=(start[0], start[1]),
                        arrowprops={
                            "arrowstyle": arrowdecision(event_type),
                            "linewidth": 2
                        })
        else:
            ax.scatter(start[0], start[1], marker=markerdecision(event_type))
            counter = 0

        ax.text(
            start[0],
            start[1] + 1,
            event["player"]["name"],
            fontsize=8
        )

        ax.set_title(
            f"{event['minute']}' - {event['player']['name']} - {event['type']['name']}"  
        ) 

    animation = FuncAnimation(
        fig,
        update,
        frames=len(animation_events),
        interval=200,
        repeat=False
    )

    animation.save(f"/workspaces/football-scout/data/animations/{match_id}.gif", writer="pillow")

    plt.close(fig)


    

In [11]:
match_id = 3749493
visualise(match_id)